# 🚀 HIGH-SPEED UPI TRANSACTION STREAMING WITH AI FRAUD DETECTION

## Features:
- ⚡ Generates 1000+ transactions per minute
- 🧠 Real-time AI Fraud Detection
- 💾 Batch database ingestion to MySQL
- 📊 Live statistics and monitoring
- 🔄 Multi-threaded processing

In [ ]:
# ============================================================================
# STEP 1: Import Libraries and Setup
# ============================================================================

import pandas as pd
import numpy as np
import random
import time
import threading
from datetime import datetime
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

In [ ]:
# ============================================================================
# STEP 2: MySQL Database Connection Setup
# ============================================================================

from sqlalchemy import create_engine, text
from urllib.parse import quote

# MySQL Configuration
DB_CONFIG = {
    'username': 'root',
    'password': 'Akash@0706',
    'host': 'localhost',
    'database': 'upi_fraud_db'
}

# Create database if not exists
def create_database():
    temp_engine = create_engine(f"mysql+pymysql://{DB_CONFIG['username']}:{quote(DB_CONFIG['password'], safe='')}@{DB_CONFIG['host']}")
    with temp_engine.connect() as conn:
        conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_CONFIG['database']}"))
        conn.commit()
    temp_engine.dispose()
    print(f"✅ Database '{DB_CONFIG['database']}' created/verified")

create_database()

# Create connection engine
engine = create_engine(f"mysql+pymysql://{DB_CONFIG['username']}:{quote(DB_CONFIG['password'], safe='')}@{DB_CONFIG['host']}/{DB_CONFIG['database']}")

print("✅ MySQL connection established")

In [ ]:
# ============================================================================
# STEP 3: Create Database Table
# ============================================================================

create_table_query = """
CREATE TABLE IF NOT EXISTS upi_transactions (
    transaction_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id VARCHAR(20),
    user_id INT,
    transaction_amount FLOAT,
    transaction_date DATETIME,
    transaction_time DATETIME,
    transaction_hour INT,
    transaction_type VARCHAR(20),
    payment_method VARCHAR(30),
    merchant_category VARCHAR(50),
    device_type VARCHAR(20),
    location VARCHAR(50),
    account_balance FLOAT,
    is_foreign_transaction INT,
    failed_login_attempts INT,
    account_tenure_years INT,
    risk_score FLOAT,
    is_fraud INT,
    behavioral_anomaly_score FLOAT,
    device_risk_score FLOAT,
    rule_based_score FLOAT,
    classification VARCHAR(20),
    triggered_rules TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    INDEX idx_customer_id (customer_id),
    INDEX idx_transaction_time (transaction_time),
    INDEX idx_is_fraud (is_fraud),
    INDEX idx_risk_score (risk_score),
    INDEX idx_classification (classification)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""

with engine.connect() as conn:
    conn.execute(text(create_table_query))
    conn.commit()

print("✅ Table 'upi_transactions' created/verified successfully")

In [ ]:
# ============================================================================
# STEP 4: AI FRAUD DETECTION SYSTEM
# ============================================================================

class CustomerBehaviorAnalyzer:
    def __init__(self):
        self.customer_profiles = defaultdict(lambda: {
            'transaction_count': 0, 'avg_amount': 0, 'total_amount': 0,
            'common_hours': defaultdict(int), 'common_locations': defaultdict(int),
            'common_devices': defaultdict(int), 'last_transaction': None
        })
    
    def analyze_behavior(self, customer_id, transaction):
        profile = self.customer_profiles[customer_id]
        if profile['transaction_count'] < 3:
            return 0
        anomaly_score = 0
        if profile['avg_amount'] > 0:
            amount_ratio = transaction['transaction_amount'] / profile['avg_amount']
            if amount_ratio > 5: anomaly_score += 25
            elif amount_ratio > 3: anomaly_score += 15
            elif amount_ratio > 2: anomaly_score += 5
        current_hour = transaction['transaction_hour']
        hour_count = profile['common_hours'].get(current_hour, 0)
        if hour_count < profile['transaction_count'] * 0.1:
            anomaly_score += 15
        if profile['common_locations'].get(transaction['location'], 0) == 0:
            anomaly_score += 20
        if profile['common_devices'].get(transaction['device_type'], 0) == 0:
            anomaly_score += 15
        if profile['last_transaction']:
            time_diff = (transaction['transaction_time'] - profile['last_transaction']).total_seconds()
            if time_diff < 60:
                anomaly_score += 10
        return min(anomaly_score, 100)
    
    def update_profile(self, customer_id, transaction):
        profile = self.customer_profiles[customer_id]
        profile['transaction_count'] += 1
        amount = transaction['transaction_amount']
        profile['total_amount'] += amount
        profile['avg_amount'] = profile['total_amount'] / profile['transaction_count']
        profile['common_hours'][transaction['transaction_hour']] += 1
        profile['common_locations'][transaction['location']] += 1
        profile['common_devices'][transaction['device_type']] += 1
        profile['last_transaction'] = transaction['transaction_time']


class FraudDetector:
    def __init__(self):
        self.behavior_analyzer = CustomerBehaviorAnalyzer()
        self.SAFE_THRESHOLD = 30
        self.REVIEW_THRESHOLD = 70
    
    def apply_rules(self, transaction):
        risk_score = 2
        triggered_rules = []
        
        amount = transaction['transaction_amount']
        hour = transaction['transaction_hour']
        device = transaction['device_type']
        is_foreign = transaction['is_foreign_transaction']
        failed_logins = transaction['failed_login_attempts']
        tenure = transaction['account_tenure_years']
        merchant = transaction['merchant_category']
        
        if amount > 50000:
            risk_score += 8
            triggered_rules.append("Very High Amount (>₹50K)")
        elif amount > 15000:
            risk_score += 4
            triggered_rules.append("High Amount (>₹15K)")
        if hour < 2:
            risk_score += 4
            triggered_rules.append("Very Unusual Time")
        elif hour < 4:
            risk_score += 2
            triggered_rules.append("Unusual Time")
        if is_foreign == 1:
            risk_score += 3
            triggered_rules.append("Foreign Transaction")
        if device in ['Web', 'Desktop']:
            risk_score += 1
            triggered_rules.append("Web Transaction")
        if failed_logins >= 5:
            risk_score += 4
            triggered_rules.append("Many Failed Logins")
        elif failed_logins >= 3:
            risk_score += 2
            triggered_rules.append("Multiple Failed Logins")
        if tenure < 1:
            risk_score += 5
            triggered_rules.append("New Account")
        if merchant in ['Electronics', 'Travel', 'Jewelry']:
            risk_score += 2
            triggered_rules.append(f"High-Risk Merchant")
        
        return min(risk_score, 100), triggered_rules
    
    def analyze_transaction(self, transaction):
        customer_id = transaction['customer_id']
        rule_score, triggered_rules = self.apply_rules(transaction)
        behavioral_score = self.behavior_analyzer.analyze_behavior(customer_id, transaction)
        self.behavior_analyzer.update_profile(customer_id, transaction)
        device_score = random.randint(0, 15) if random.random() < 0.1 else 0
        final_score = (rule_score * 0.60) + (behavioral_score * 0.25) + (device_score * 0.15)
        final_score = min(final_score, 100)
        
        if final_score > self.REVIEW_THRESHOLD:
            classification = "BLOCK"
        elif final_score > self.SAFE_THRESHOLD:
            classification = "REVIEW"
        else:
            classification = "APPROVE"
        
        return {
            'risk_score': round(final_score, 2),
            'is_fraud': 1 if classification == "BLOCK" else 0,
            'behavioral_anomaly_score': behavioral_score,
            'device_risk_score': device_score,
            'rule_based_score': rule_score,
            'classification': classification,
            'triggered_rules': ", ".join(triggered_rules) if triggered_rules else "None"
        }


# Initialize fraud detector
fraud_detector = FraudDetector()
print("✅ AI Fraud Detection System initialized")

In [ ]:
# ============================================================================
# STEP 5: Transaction Generator & Configuration
# ============================================================================

def generate_transaction():
    customer_id = f"C{random.randint(1000, 9999)}"
    user_id = random.randint(1, 5000)
    
    if random.random() < 0.1:
        transaction_amount = round(random.uniform(15000, 100000), 2)
    else:
        transaction_amount = round(np.random.exponential(scale=3000), 2)
        transaction_amount = max(transaction_amount, 10)
    
    transaction_type = random.choice(['Debit', 'Credit'])
    payment_method = random.choice(['Card', 'NetBanking', 'UPI', 'Wallet'])
    merchant_category = random.choice(['Electronics', 'Grocery', 'Travel', 'Restaurant', 'Healthcare', 'Entertainment', 'Utilities', 'Shopping'])
    device_type = random.choice(['Android', 'iOS', 'Web', 'Mobile', 'Desktop'])
    location = random.choice(['Chennai', 'Mumbai', 'Delhi', 'Hyderabad', 'Bangalore', 'Pune', 'Kolkata', 'Chandigarh', 'Ahmedabad', 'Jaipur'])
    account_balance = round(random.uniform(5000, 500000), 2)
    is_foreign_transaction = random.choice([0, 1])
    failed_login_attempts = random.randint(0, 5)
    account_tenure_years = random.randint(0, 20)
    
    now = datetime.now()
    
    return {
        'customer_id': customer_id,
        'user_id': user_id,
        'transaction_amount': transaction_amount,
        'transaction_date': now,
        'transaction_time': now,
        'transaction_hour': now.hour,
        'transaction_type': transaction_type,
        'payment_method': payment_method,
        'merchant_category': merchant_category,
        'device_type': device_type,
        'location': location,
        'account_balance': account_balance,
        'is_foreign_transaction': is_foreign_transaction,
        'failed_login_attempts': failed_login_attempts,
        'account_tenure_years': account_tenure_years
    }

# Configuration
TARGET_TXNS_PER_SECOND = 20
BATCH_SIZE = 50
STREAMING_DURATION = 0

# Statistics tracking
stats = {
    'total_generated': 0, 'total_inserted': 0, 'fraud_detected': 0,
    'safe_transactions': 0, 'review_transactions': 0, 'blocked_transactions': 0,
    'start_time': None, 'batch_count': 0
}

transaction_buffer = []
buffer_lock = threading.Lock()

print("✅ Transaction generator configured")
print(f"📊 Target: {TARGET_TXNS_PER_SECOND} transactions/second")
print(f"📦 Batch size: {BATCH_SIZE} transactions")

In [ ]:
# ============================================================================
# STEP 6: Database Ingestion Thread
# ============================================================================

def batch_insert_to_db():
    global transaction_buffer, stats
    
    while True:
        try:
            time.sleep(0.5)
            with buffer_lock:
                if len(transaction_buffer) >= BATCH_SIZE:
                    batch = transaction_buffer[:BATCH_SIZE]
                    transaction_buffer = transaction_buffer[BATCH_SIZE:]
            
            if batch:
                df = pd.DataFrame(batch)
                df.to_sql('upi_transactions', con=engine, if_exists='append', index=False)
                stats['total_inserted'] += len(batch)
                stats['batch_count'] += 1
                fraud_count = sum(1 for t in batch if t['is_fraud'] == 1)
                stats['fraud_detected'] += fraud_count
        except Exception as e:
            print(f"⚠ DB Insert Error: {e}")
            time.sleep(1)

db_thread = threading.Thread(target=batch_insert_to_db, daemon=True)
db_thread.start()

print("✅ Database ingestion thread started")

In [ ]:
# ============================================================================
# STEP 7: HIGH-SPEED TRANSACTION STREAMING (MAIN LOOP)
# ============================================================================

def start_streaming():
    global stats, transaction_buffer
    
    stats['start_time'] = datetime.now()
    
    print("\n" + "="*80)
    print("🚀 STARTING HIGH-SPEED UPI TRANSACTION STREAMING")
    print("="*80)
    print(f"📊 Target: {TARGET_TXNS_PER_SECOND} transactions/second")
    print(f"📦 Batch size: {BATCH_SIZE} transactions")
    print("🧠 AI Fraud Detection: ACTIVE")
    print("💾 Database: INSERTING")
    print("\n⏹ Press STOP button or interrupt kernel to stop streaming")
    print("="*80 + "\n")
    
    try:
        while True:
            start_time = time.time()
            
            for _ in range(TARGET_TXNS_PER_SECOND):
                transaction = generate_transaction()
                fraud_result = fraud_detector.analyze_transaction(transaction)
                transaction.update(fraud_result)
                
                stats['total_generated'] += 1
                
                if transaction['classification'] == 'BLOCK':
                    stats['blocked_transactions'] += 1
                elif transaction['classification'] == 'REVIEW':
                    stats['review_transactions'] += 1
                else:
                    stats['safe_transactions'] += 1
                
                with buffer_lock:
                    transaction_buffer.append(transaction)
            
            elapsed = time.time() - start_time
            sleep_time = max(0, 1.0 - elapsed)
            if sleep_time > 0:
                time.sleep(sleep_time)
            
            elapsed_total = (datetime.now() - stats['start_time']).total_seconds()
            rate = stats['total_generated'] / elapsed_total if elapsed_total > 0 else 0
            
            print(f"\r⏱ [{datetime.now().strftime('%H:%M:%S')}] "
                  f"Total: {stats['total_generated']:,} | "
                  f"Rate: {rate:.1f}/sec | "
                  f"In DB: {stats['total_inserted']:,} | "
                  f"Blocked: {stats['blocked_transactions']:,} | "
                  f"Buffer: {len(transaction_buffer)}", 
                  end='', flush=True)
            
            if STREAMING_DURATION > 0 and elapsed_total >= STREAMING_DURATION:
                break
                
    except KeyboardInterrupt:
        print("\n\n⚠️ Streaming interrupted by user")
    
    return stats

# Start streaming (this will run until interrupted)
final_stats = start_streaming()

In [ ]:
# ============================================================================
# STEP 8: FINAL STATISTICS SUMMARY
# ============================================================================

time.sleep(2)

print("\n" + "="*80)
print("📊 STREAMING SESSION SUMMARY")
print("="*80)

elapsed = (datetime.now() - stats['start_time']).total_seconds() if stats['start_time'] else 0

print(f"\n⏱ Session Duration: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"📈 Total Transactions Generated: {stats['total_generated']:,}")
print(f"💾 Total Inserted to Database: {stats['total_inserted']:,}")
print(f"📊 Average Rate: {stats['total_generated']/elapsed:.2f} transactions/second")
print(f"\n🚫 Blocked (Fraud): {stats['blocked_transactions']:,} ({stats['blocked_transactions']/stats['total_generated']*100:.2f}%)")
print(f"⚠️  Review Required: {stats['review_transactions']:,} ({stats['review_transactions']/stats['total_generated']*100:.2f}%)")
print(f"✅ Safe Transactions: {stats['safe_transactions']:,} ({stats['safe_transactions']/stats['total_generated']*100:.2f}%)")
print("\n" + "="*80)

In [ ]:
# ============================================================================
# STEP 9: VERIFY DATA IN DATABASE
# ============================================================================

print("\n📋 Verifying data in database...")

try:
    total_df = pd.read_sql("SELECT COUNT(*) as total FROM upi_transactions", engine)
    total = total_df.iloc[0]['total']
    
    fraud_df = pd.read_sql("""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) as fraud_count,
            SUM(CASE WHEN classification = 'BLOCK' THEN 1 ELSE 0 END) as blocked,
            SUM(CASE WHEN classification = 'REVIEW' THEN 1 ELSE 0 END) as review,
            SUM(CASE WHEN classification = 'APPROVE' THEN 1 ELSE 0 END) as approved,
            AVG(risk_score) as avg_risk_score,
            MAX(risk_score) as max_risk_score,
            SUM(transaction_amount) as total_amount
        FROM upi_transactions
    """, engine)
    
    print("\n" + "="*80)
    print("💾 DATABASE VERIFICATION RESULTS")
    print("="*80)
    print(f"\n📊 Total Transactions in DB: {total:,}")
    print(f"🚫 Blocked (Fraud): {fraud_df.iloc[0]['blocked']:,}")
    print(f"⚠️  Review: {fraud_df.iloc[0]['review']:,}")
    print(f"✅ Approved: {fraud_df.iloc[0]['approved']:,}")
    print(f"📈 Average Risk Score: {fraud_df.iloc[0]['avg_risk_score']:.2f}%")
    print(f"💰 Total Transaction Amount: ₹{fraud_df.iloc[0]['total_amount']:,.2f}")
    
    latest_df = pd.read_sql("""
        SELECT 
            transaction_id, customer_id, transaction_amount, 
            risk_score, classification, device_type, location
        FROM upi_transactions 
        ORDER BY transaction_id DESC 
        LIMIT 10
    """, engine)
    
    print("\n📌 Latest 10 Transactions:")
    print(latest_df.to_string())
    
except Exception as e:
    print(f"⚠ Error verifying database: {e}")

print("\n" + "="*80)

In [ ]:
# ============================================================================
# STEP 10: ANALYTICS AND VISUALIZATIONS
# ============================================================================

import matplotlib.pyplot as plt


try:
    viz_df = pd.read_sql("""
        SELECT 
            HOUR(transaction_time) as hour,
            COUNT(*) as count,
            SUM(is_fraud) as fraud_count,
            AVG(risk_score) as avg_risk
        FROM upi_transactions 
        GROUP BY HOUR(transaction_time)
        ORDER BY hour
    """, engine)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('UPI Transaction Fraud Detection Analytics', fontsize=16, fontweight='bold')

    axes[0, 0].bar(viz_df['hour'], viz_df['count'], color='steelblue', alpha=0.7)
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Transaction Count')
    axes[0, 0].set_title('Transactions by Hour')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].bar(viz_df['hour'], viz_df['count'] - viz_df['fraud_count'], label='Safe', color='green', alpha=0.7)
    axes[0, 1].bar(viz_df['hour'], viz_df['fraud_count'], bottom=viz_df['count'] - viz_df['fraud_count'], label='Fraud', color='red', alpha=0.7)
    axes[0, 1].set_xlabel('Hour of Day')
    axes[0, 1].set_ylabel('Transaction Count')
    axes[0, 1].set_title('Fraud vs Safe Transactions by Hour')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(viz_df['hour'], viz_df['avg_risk'], marker='o', color='orange', linewidth=2)
    axes[1, 0].set_xlabel('Hour of Day')
    axes[1, 0].set_ylabel('Average Risk Score (%)')
    axes[1, 0].set_title('Average Risk Score by Hour')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(y=70, color='red', linestyle='--', label='Block Threshold')
    axes[1, 0].axhline(y=30, color='yellow', linestyle='--', label='Review Threshold')
    axes[1, 0].legend()

    class_df = pd.read_sql("SELECT classification, COUNT(*) as count FROM upi_transactions GROUP BY classification", engine)
    colors = ['green', 'orange', 'red']
    axes[1, 1].pie(class_df['count'], labels=class_df['classification'], autopct='%1.1f%%', colors=colors, startangle=90)
    axes[1, 1].set_title('Transaction Classification Distribution')

    plt.tight_layout()
    plt.show()
    
    print("✅ Analytics visualizations generated")
    
except Exception as e:
    print(f"⚠ Error generating visualizations: {e}")

## 🎯 How to Use

### Run High-Speed Streaming:
1. Run cells 1-7 to setup the system
2. Run cell 7 (STEP 7) to start streaming
3. Watch transactions being generated in real-time
4. Press Stop button or Interrupt Kernel to stop
5. Run cells 8-10 to see statistics and analytics

### Performance:
- ⚡ **20 transactions/second** (1200/minute)
- 🧠 **AI Fraud Detection** with behavioral analysis
- 💾 **Batch database insertion** for efficiency
- 📊 **Real-time statistics** and monitoring